In [1]:
!nvidia-smi

Wed Nov  4 03:52:08 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   56C    P8             12W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!git clone https://github.com/amaniqubel1/assessment-2-python.git
%cd /content/assessment-2-python
!pip install -q "transformers<5" ujson tqdm gitpython rouge-score sacrebleu nltk sentencepiece pytorch-lightning wandb gdown

Cloning into 'assessment-2-python'...
remote: Enumerating objects: 240, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (96/96), done.
remote: Total 240 (delta 49), reused 66 (delta 20), pack-reused 122 (from 1)
Receiving objects: 100% (240/240), 18.37 MiB | 17.97 MiB/s, done.
Resolving deltas: 100% (77/77), done.
/content/assessment-2-python
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio-client 2.7.2 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
diffusers 0.42.1 requires huggingface-hub<3.0,>=1.32.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.31.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggin

In [3]:
!python -c "from huggingface_hub import snapshot_download; snapshot_download(repo_id='Jiangjie/loren', local_dir='models', allow_patterns=['fact_checking/roberta-large/*'])"
!ls -la models/fact_checking/roberta-large

Fetching 7 files:   0% 0/7 [00:00<?, ?it/s]
fact_checking/roberta-large/pytorch_mode(…):   0% 0.00/1.43G [00:00<?, ?B/s]

fact_checking/roberta-large/training_arg(…):   0% 0.00/1.77k [00:00<?, ?B/s]


merges.txt: 456kB [00:00, 77.8MB/s]



special_tokens_map.json: 100% 772/772 [00:00<00:00, 7.07MB/s]



tokenizer_config.json: 1.05kB [00:00, 7.92MB/s]



vocab.json: 0.00B [00:00, ?B/s]



config.json: 100% 703/703 [00:00<00:00, 5.35MB/s]
vocab.json: 899kB [00:00, 72.7MB/s]
Fetching 7 files:  14% 1/7 [00:01<00:06,  1.05s/it]

fact_checking/roberta-large/training_arg(…): 100% 1.77k/1.77k [00:02<00:00, 612B/s]

fact_checking/roberta-large/pytorch_mode(…):  14% 201M/1.43G [00:08<00:50, 24.3MB/s]
fact_checking/roberta-large/pytorch_mode(…):  19% 268M/1.43G [00:08<00:33, 34.3MB/s]
fact_checking/roberta-large/pytorch_mode(…):  24% 335M/1.43G [00:12<00:39, 27.5MB/s]
fact_checking/roberta-large/pytorch_mode(…):  28% 402M/1.43G [00:12<00:26, 38.1MB/s]
fact_checking/roberta-large/pytorch_mode(…): 

In [4]:
!gdown 1kZxHidaDCe5GWMIuNa5fVeK4LPYCpvDD -O data.zip
!unzip -q data.zip -d data/
!cp -r data/data/* data/
!ls -la data/fact_checking/v5

Downloading...
From (original): https://drive.google.com/uc?id=1kZxHidaDCe5GWMIuNa5fVeK4LPYCpvDD
From (redirected): https://drive.google.com/uc?id=1kZxHidaDCe5GWMIuNa5fVeK4LPYCpvDD&confirm=t&uuid=e3877095-afc6-40ca-a985-64e3f99c0119
To: /content/assessment-2-python/data.zip
100% 670M/670M [00:07<00:00, 86.0MB/s]
total 524380
drwxr-xr-x 2 root root      4096 Nov  4 03:54 .
drwxr-xr-x 3 root root      4096 Nov  4 03:54 ..
-rw-r--r-- 1 root root      6148 Nov  4 03:54 .DS_Store
-rw-r--r-- 1 root root  54212889 Nov  4 03:54 eval.json
-rw-r--r-- 1 root root  56182795 Nov  4 03:54 test.json
-rw-r--r-- 1 root root 375142016 Nov  4 03:54 train.json
-rw-r--r-- 1 root root  51401255 Nov  4 03:54 val.json


In [5]:
!sed -i 's/from transformers.file_utils import cached_property/from functools import cached_property/' src/mrc_client/seq2seq/seq2seq_utils.py
!sed -i 's/^from transformers import WEIGHTS_NAME, AdamW, get_linear_schedule_with_warmup/from transformers import get_linear_schedule_with_warmup\nfrom torch.optim import AdamW\nWEIGHTS_NAME = "pytorch_model.bin"/' src/check_client/train.py
!sed -i "s/parser.add_argument('--n_gpu', default=4)/parser.add_argument('--n_gpu', default=1, type=int)/" src/check_client/fact_checker.py
!sed -i 's/self.init_weights()/self.post_init()/' src/check_client/plm_checkers/roberta_checker.py src/check_client/plm_checkers/bert_checker.py
!grep -n "post_init" src/check_client/plm_checkers/*.py
!grep -n "n_gpu', default" src/check_client/fact_checker.py

src/check_client/plm_checkers/bert_checker.py:44:        self.post_init()
src/check_client/plm_checkers/roberta_checker.py:49:        self.post_init()
182:    parser.add_argument('--n_gpu', default=1, type=int)


In [6]:
!cd data/fact_checking/v5 && cp eval.json eval_full.json && head -n 1000 eval_full.json > eval.json && wc -l eval.json

1000 eval.json


In [13]:
%cd /content/assessment-2-python/src/check_client
!PJ_HOME=/content/assessment-2-python/ python fact_checker.py -i eval -c /content/assessment-2-python/models/fact_checking/roberta-large --model_type roberta --model_name_or_path roberta-large --prior nli -l 0.5 -k 3 -b 16 --n_gpu 1 -v

/content/assessment-2-python/src/check_client
I0000 00:00:1793765254.934415    4799 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
Converting Examples: 100% 1000/1000 [00:04<00:00, 205.73it/s]
Fact Checking: 100% 63/63 [03:50<00:00,  3.66s/it]
91198	Colin Kaepernick became a starting quarterback during the 49ers 63rd season in the National Football League .	1
194462	Tilda Swinton is a vegan .	1
137334	Fox 2000 Pictures released the film Soul Food .	2
166626	Anne Rice was born in New Jersey .	0
111897	Telemundo is a English - language television network .	0
{'SUPPORTS Precision': np.float64(0.8463611859838275), 'SUPPORTS Recall': np.float64(0.9486404833836858), 'SUPPORTS F1': np.float64(0.8945868945868946), 'REFUTES Precision': np.float64(0.8102409638554217), 'REFUTES Recall': np.float64(0.7935103244837758), 'REFUTES F1': np.float64(0.8017883755588674), 'NOT ENOUGH INFO Precision': np.float64(0.7676767676767676), 'NOT ENOUGH INFO Recall': np.float6

In [14]:
import os, sys
os.environ['PJ_HOME']='/content/assessment-2-python/'
sys.path.append('/content/assessment-2-python/src')
sys.path.append('/content/assessment-2-python/src/eval_client')
from eval_client.fever_scorer import FeverScorer

pred='/content/assessment-2-python/results/fact_checking/AAAI22/eval.roberta-large_m0.0_l0.5_roberta-large_nli.predictions.jsonl'
res=FeverScorer().get_scores(pred)
print("=== LOREN results (RoBERTa-large, first 1000 eval claims) ===")
for k,v in res.items():
    print(f"{k}: {float(v):.4f}")

=== LOREN results (RoBERTa-large, first 1000 eval claims) ===
SUPPORTS Precision: 0.8464
SUPPORTS Recall: 0.9486
SUPPORTS F1: 0.8946
REFUTES Precision: 0.8102
REFUTES Recall: 0.7935
REFUTES F1: 0.8018
NOT ENOUGH INFO Precision: 0.7677
NOT ENOUGH INFO Recall: 0.6909
NOT ENOUGH INFO F1: 0.7273
Evidence Precision: 0.2776
Evidence Recall: 0.9627
Evidence F1: 0.4309
FEVER Score: 0.7910
Label Accuracy: 0.8110


In [8]:
!find /content/assessment-2-python/results -name "*predictions*" -exec ls -la {} \; -exec head -c 1500 {} \; -exec wc -l {} \;

-rw-r--r-- 1 root root 10137428 Nov  4 03:52 /content/assessment-2-python/results/AAAI22/test.bert-large-cased_m0.0_l0.5_checkpoint-4200_random.predictions.jsonl
{"id":89296,"predicted_label":"REFUTES","predicted_evidence":[["Henry_Spencer_-LRB-Psych-RRB-",1],["Henry_Spencer_-LRB-Psych-RRB-",0],["Henry_Spencer",0],["Henry_Spencer",12],["Henry_Spencer",13]],"z_prob":[[0.007646209094673395,0.01507676113396883,0.9772770404815674],[0.6997641324996948,0.288274347782135,0.011961492709815502],[0.16477805376052856,0.4959378242492676,0.3392840325832367],[0.007619157433509827,0.014660353772342205,0.9777204990386963]],"m_attn":[0.008005472831428051,0.9381691813468933,0.04584575444459915,0.007979576475918293]}
{"id":78554,"predicted_label":"REFUTES","predicted_evidence":[["John_Ritter",0],["John_Ritter",8],["John_Ritter",9],["John_Ritter",3],["John_Ritter",4]],"z_prob":[[0.02725931443274021,0.01027725264430046,0.9624634385108948],[0.8440686464309692,0.13638576865196228,0.01954558491706848],[0.0613

In [9]:
import json
p='/content/assessment-2-python/results/fact_checking/AAAI22/eval.roberta-large_m0.0_l0.5_roberta-large_nli.predictions.jsonl'
rows=[json.loads(l) for l in open(p)]
print(len(rows), sum('predicted_label' in r for r in rows))

1000 1000


In [10]:
!sed -n '1,70p' /content/assessment-2-python/src/eval_client/fever_scorer.py


# -*- coding:utf-8  -*-

"""
@Author             : Bao
@Date               : 2020/8/24
@Desc               :
@Last modified by   : Bao
@Last modified date : 2020/9/1
"""

import os
import json
import numpy as np
from collections import defaultdict
import tensorflow as tf
from sklearn.metrics import precision_recall_fscore_support
try:
    from .scorer import fever_score
except:
    from scorer import fever_score


prefix = os.environ['PJ_HOME']


class FeverScorer:
    def __init__(self):
        self.id2label = {2: 'SUPPORTS', 0: 'REFUTES', 1: 'NOT ENOUGH INFO'}
        self.label2id = {value: key for key, value in self.id2label.items()}

    def get_scores(self, predicted_file, actual_file=f'{prefix}/data/fever/shared_task_dev.jsonl'):
        id2results = defaultdict(dict)

        with tf.io.gfile.GFile(predicted_file) as f:
            for line in f:
                js = json.loads(line)
                guid = js['id']
                id2results[guid] = js

        with tf.io.gfil

In [11]:
p='/content/assessment-2-python/src/eval_client/fever_scorer.py'
s=open(p).read()
old="                id2results[guid]['evidence'] = evidence\n                id2results[guid]['label'] = label\n"
new="                if guid not in id2results:\n                    continue\n                id2results[guid]['evidence'] = evidence\n                id2results[guid]['label'] = label\n"
assert old in s, "pattern not found"
open(p,'w').write(s.replace(old,new))
print("patched")


patched


In [12]:
import os, sys
os.environ['PJ_HOME']='/content/assessment-2-python/'
sys.path.append('/content/assessment-2-python/src')
sys.path.append('/content/assessment-2-python/src/eval_client')
from eval_client.fever_scorer import FeverScorer

pred='/content/assessment-2-python/results/fact_checking/AAAI22/eval.roberta-large_m0.0_l0.5_roberta-large_nli.predictions.jsonl'
res=FeverScorer().get_scores(pred)
for k,v in res.items():
    print(f"{k}: {float(v):.4f}")

SUPPORTS Precision: 0.8464
SUPPORTS Recall: 0.9486
SUPPORTS F1: 0.8946
REFUTES Precision: 0.8102
REFUTES Recall: 0.7935
REFUTES F1: 0.8018
NOT ENOUGH INFO Precision: 0.7677
NOT ENOUGH INFO Recall: 0.6909
NOT ENOUGH INFO F1: 0.7273
Evidence Precision: 0.2776
Evidence Recall: 0.9627
Evidence F1: 0.4309
FEVER Score: 0.7910
Label Accuracy: 0.8110
